# James Check "Buy the Dip" Signal Analysis

## Overview
This notebook implements and tests James Check's "Buy the Dip" framework from his Checkmate masterclass.

### Core Conditions (Swiss Army Knife):
1. **STH-MVRV < 1.0** - Short-term holders underwater (cost basis > price)
2. **STH-SOPR < 1.0** - Short-term holders selling at a loss

### Optional Enhancement Conditions:
3. **Price > 200-day SMA** - Bull market filter
4. **Funding Rates Negative/Low** - Derivatives reset (if available)
5. **High Liquidations** - Leverage flush (if available)

### Comparison Signals:
- Existing approved signals (NVT, MVRV, MVRV_LTH)
- Individual STH-MVRV and STH-SOPR signals

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Project imports
from src.backtester import Backtester, Signal, BacktestConfig, ExitMode
from src.config import BULL_MARKETS, BEAR_MARKETS, DATA_DIR

# Data directories
BRK_DATA_DIR = DATA_DIR / 'brk' / 'daily'
GLASSNODE_DATA_DIR = DATA_DIR / 'glassnode' / 'daily'

# Styling
plt.style.use('dark_background')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 10

print(f"BRK Data Dir: {BRK_DATA_DIR}")
print(f"Glassnode Data Dir: {GLASSNODE_DATA_DIR}")
print("\nImports complete!")

## 1. Load Data

In [ ]:
def load_metrics_from_dir(data_dir: Path) -> pd.DataFrame:
    """Load all metrics from a directory into a single DataFrame.
    
    Handles two data formats:
    - BRK format: 'time' as column, needs set_index
    - Glassnode format: 'time' already as index
    """
    if not data_dir.exists():
        print(f"Directory not found: {data_dir}")
        return pd.DataFrame()
    
    dfs = {}
    for f in sorted(data_dir.glob("*.parquet")):
        metric_name = f.stem
        temp = pd.read_parquet(f)
        
        # Handle both formats: time as column or time as index
        if 'time' in temp.columns:
            temp = temp.set_index('time')
        elif temp.index.name != 'time':
            print(f"  Warning: {metric_name} has no 'time' column or index, skipping")
            continue
            
        temp = temp.rename(columns={'value': metric_name})
        dfs[metric_name] = temp
    
    if dfs:
        combined = pd.concat(dfs.values(), axis=1)
        return combined.sort_index()
    return pd.DataFrame()

# Load BRK on-chain data
df = load_metrics_from_dir(BRK_DATA_DIR)
print(f"Loaded {len(df.columns)} BRK metrics")

# Load Glassnode derivatives data (if available)
df_deriv = load_metrics_from_dir(GLASSNODE_DATA_DIR)

# Merge if derivatives available
if not df_deriv.empty:
    df = df.join(df_deriv, how='left')
    print(f"✓ Added derivatives data: {list(df_deriv.columns)}")
else:
    print("⚠ No derivatives data found")

# ===== DATA VALIDATION =====
# Remove duplicate indices (can happen with timezone mismatches)
if df.index.duplicated().any():
    n_dups = df.index.duplicated().sum()
    print(f"⚠ Removing {n_dups} duplicate index entries")
    df = df[~df.index.duplicated(keep='last')]

# Ensure sorted index
df = df.sort_index()

# Verify no duplicates
assert not df.index.duplicated().any(), "Duplicate indices remain!"

print(f"\nDataset: {len(df)} rows, {len(df.columns)} columns")
print(f"Date range: {df.index.min()} to {df.index.max()}")
print(f"\nKey metrics available:")
key_metrics = ['price', 'mvrv_sth', 'sopr_sth', 'realized_price_sth', 'mvrv', 'nupl', 'price_200d_sma']
for m in key_metrics:
    if m in df.columns:
        print(f"  ✓ {m}")
    else:
        print(f"  ✗ {m} (MISSING)")

print(f"\n✓ Data validation passed")

In [ ]:
# Preview key metrics
display_cols = ['price', 'mvrv_sth', 'sopr_sth', 'realized_price_sth', 'mvrv', 'nupl']
available_cols = [c for c in display_cols if c in df.columns]
df[available_cols].tail(10)

## 2. Calculate James Check Buy-the-Dip Signal

### Signal Logic:
**Core (Required):**
- STH-MVRV < 1.0 (short-term holders underwater)
- STH-SOPR < 1.0 (short-term holders selling at a loss)

**Enhanced (Optional):**
- Price > 200-day SMA (bull market context)
- Funding Rate < 0.01% (derivatives reset)

In [ ]:
# Create James Check signals
df_signals = df.copy()

# Core conditions
df_signals['sth_mvrv_below_1'] = df_signals['mvrv_sth'] < 1.0
df_signals['sth_sopr_below_1'] = df_signals['sopr_sth'] < 1.0

# Bull market filter (Price > 200 SMA)
if 'price_200d_sma' in df_signals.columns:
    df_signals['bull_filter'] = df_signals['price'] > df_signals['price_200d_sma']
else:
    # Calculate 200-day SMA if not available
    df_signals['price_200d_sma'] = df_signals['price'].rolling(200).mean()
    df_signals['bull_filter'] = df_signals['price'] > df_signals['price_200d_sma']

# Derivatives filter (if available)
if 'funding_rate' in df_signals.columns:
    df_signals['funding_reset'] = df_signals['funding_rate'] < 0.0001  # < 0.01%
else:
    df_signals['funding_reset'] = True  # No filter if unavailable

# ===== SIGNAL VARIANTS =====

# 1. Core: STH-MVRV AND STH-SOPR both < 1.0
df_signals['jc_core'] = (
    df_signals['sth_mvrv_below_1'] & 
    df_signals['sth_sopr_below_1']
)

# 2. Enhanced: Core + Bull market filter
df_signals['jc_enhanced'] = (
    df_signals['jc_core'] & 
    df_signals['bull_filter']
)

# 3. Full: Enhanced + Derivatives filter
df_signals['jc_full'] = (
    df_signals['jc_enhanced'] & 
    df_signals['funding_reset']
)

# 4. Individual components for comparison
df_signals['sth_mvrv_only'] = df_signals['sth_mvrv_below_1']
df_signals['sth_sopr_only'] = df_signals['sth_sopr_below_1']

# Summary
print("=" * 60)
print("JAMES CHECK BUY-THE-DIP SIGNAL SUMMARY")
print("=" * 60)
signals = ['sth_mvrv_only', 'sth_sopr_only', 'jc_core', 'jc_enhanced', 'jc_full']
for s in signals:
    n_signals = df_signals[s].sum()
    pct = n_signals / len(df_signals) * 100
    print(f"{s:<20}: {n_signals:>5} days ({pct:.1f}%)")

print(f"\nCurrent Status (Latest):")
latest = df_signals.iloc[-1]
print(f"  STH-MVRV: {latest['mvrv_sth']:.3f} (< 1.0: {latest['sth_mvrv_below_1']})")
print(f"  STH-SOPR: {latest['sopr_sth']:.3f} (< 1.0: {latest['sth_sopr_below_1']})")
print(f"  Bull Filter: {latest['bull_filter']}")
print(f"  JC Core Signal: {latest['jc_core']}")
print(f"  JC Enhanced Signal: {latest['jc_enhanced']}")

## 3. Exploratory Statistics

Before backtesting, let's analyze the signal's statistical properties:
- Forward returns when signal is active
- Correlation with future price movements
- Distribution analysis

In [ ]:
# Calculate forward returns
for days in [7, 14, 30, 60, 90]:
    df_signals[f'fwd_ret_{days}d'] = df_signals['price'].pct_change(days).shift(-days)

# Filter to valid data (remove NaNs)
df_valid = df_signals.dropna(subset=['mvrv_sth', 'sopr_sth', 'fwd_ret_30d'])

print("=" * 70)
print("FORWARD RETURNS WHEN SIGNAL IS ACTIVE")
print("=" * 70)

# Compare returns: signal active vs not active
for signal_name in ['jc_core', 'jc_enhanced', 'sth_mvrv_only', 'sth_sopr_only']:
    signal_active = df_valid[df_valid[signal_name] == True]
    signal_inactive = df_valid[df_valid[signal_name] == False]
    
    print(f"\n{signal_name.upper()}")
    print("-" * 50)
    print(f"{'Period':<12} {'Active':>12} {'Inactive':>12} {'Diff':>10}")
    print("-" * 50)
    
    for days in [7, 14, 30, 60, 90]:
        col = f'fwd_ret_{days}d'
        active_ret = signal_active[col].mean() * 100
        inactive_ret = signal_inactive[col].mean() * 100
        diff = active_ret - inactive_ret
        print(f"{days}d:{'':<7} {active_ret:>+10.1f}%  {inactive_ret:>+10.1f}%  {diff:>+8.1f}%")

In [ ]:
# Statistical tests
from scipy import stats

print("\n" + "=" * 70)
print("STATISTICAL SIGNIFICANCE TESTS (30-day forward returns)")
print("=" * 70)

for signal_name in ['jc_core', 'jc_enhanced', 'sth_mvrv_only']:
    active = df_valid[df_valid[signal_name] == True]['fwd_ret_30d'].dropna()
    inactive = df_valid[df_valid[signal_name] == False]['fwd_ret_30d'].dropna()
    
    # T-test
    t_stat, p_value = stats.ttest_ind(active, inactive)
    
    # Mann-Whitney U (non-parametric)
    u_stat, u_pvalue = stats.mannwhitneyu(active, inactive, alternative='greater')
    
    print(f"\n{signal_name.upper()}")
    print(f"  N (active/inactive): {len(active)} / {len(inactive)}")
    print(f"  Mean return: {active.mean()*100:.2f}% vs {inactive.mean()*100:.2f}%")
    print(f"  T-test p-value: {p_value:.4f} {'✓ Significant' if p_value < 0.05 else '✗ Not Significant'}")
    print(f"  Mann-Whitney p-value: {u_pvalue:.4f} {'✓ Significant' if u_pvalue < 0.05 else '✗ Not Significant'}")

In [ ]:
# Visualize signal and forward returns distribution
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. STH-MVRV distribution
ax1 = axes[0, 0]
ax1.hist(df_valid['mvrv_sth'].dropna(), bins=50, alpha=0.7, color='cyan', edgecolor='white')
ax1.axvline(1.0, color='red', linestyle='--', linewidth=2, label='Threshold = 1.0')
ax1.set_xlabel('STH-MVRV')
ax1.set_ylabel('Frequency')
ax1.set_title('STH-MVRV Distribution')
ax1.legend()

# 2. STH-SOPR distribution
ax2 = axes[0, 1]
ax2.hist(df_valid['sopr_sth'].dropna(), bins=50, alpha=0.7, color='orange', edgecolor='white')
ax2.axvline(1.0, color='red', linestyle='--', linewidth=2, label='Threshold = 1.0')
ax2.set_xlabel('STH-SOPR')
ax2.set_ylabel('Frequency')
ax2.set_title('STH-SOPR Distribution')
ax2.legend()

# 3. Forward returns comparison (box plot)
ax3 = axes[1, 0]
data_to_plot = [
    df_valid[df_valid['jc_core'] == True]['fwd_ret_30d'].dropna() * 100,
    df_valid[df_valid['jc_core'] == False]['fwd_ret_30d'].dropna() * 100
]
bp = ax3.boxplot(data_to_plot, labels=['Signal Active', 'Signal Inactive'], patch_artist=True)
bp['boxes'][0].set_facecolor('green')
bp['boxes'][1].set_facecolor('gray')
ax3.axhline(0, color='white', linestyle='--', alpha=0.5)
ax3.set_ylabel('30-day Forward Return (%)')
ax3.set_title('JC Core Signal: Forward Return Distribution')

# 4. Win rate by holding period
ax4 = axes[1, 1]
periods = [7, 14, 30, 60, 90]
win_rates_active = []
win_rates_inactive = []

for days in periods:
    col = f'fwd_ret_{days}d'
    active_wins = (df_valid[df_valid['jc_core']][col] > 0).mean() * 100
    inactive_wins = (df_valid[~df_valid['jc_core']][col] > 0).mean() * 100
    win_rates_active.append(active_wins)
    win_rates_inactive.append(inactive_wins)

x = np.arange(len(periods))
width = 0.35
ax4.bar(x - width/2, win_rates_active, width, label='JC Core Active', color='green')
ax4.bar(x + width/2, win_rates_inactive, width, label='JC Core Inactive', color='gray')
ax4.axhline(50, color='red', linestyle='--', alpha=0.5)
ax4.set_xlabel('Holding Period (days)')
ax4.set_ylabel('Win Rate (%)')
ax4.set_title('Win Rate by Holding Period')
ax4.set_xticks(x)
ax4.set_xticklabels([f'{p}d' for p in periods])
ax4.legend()

plt.tight_layout()
plt.savefig('../data/results/jc_signal_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Correlation Analysis

In [ ]:
# Calculate correlation between metrics and forward returns
print("=" * 70)
print("CORRELATION WITH FORWARD RETURNS")
print("=" * 70)

metrics_to_test = ['mvrv_sth', 'sopr_sth', 'mvrv', 'nupl', 'sell_side_risk_sth']
metrics_available = [m for m in metrics_to_test if m in df_valid.columns]

corr_results = []
for metric in metrics_available:
    valid_data = df_valid[[metric, 'fwd_ret_7d', 'fwd_ret_30d', 'fwd_ret_60d']].dropna()
    
    for days in [7, 30, 60]:
        col = f'fwd_ret_{days}d'
        # Pearson correlation
        pearson_r, pearson_p = stats.pearsonr(valid_data[metric], valid_data[col])
        # Spearman correlation (rank-based, more robust)
        spearman_r, spearman_p = stats.spearmanr(valid_data[metric], valid_data[col])
        
        corr_results.append({
            'metric': metric,
            'period': f'{days}d',
            'pearson_r': pearson_r,
            'pearson_p': pearson_p,
            'spearman_r': spearman_r,
            'spearman_p': spearman_p
        })

corr_df = pd.DataFrame(corr_results)
display(corr_df.style.format({
    'pearson_r': '{:.3f}',
    'pearson_p': '{:.4f}',
    'spearman_r': '{:.3f}',
    'spearman_p': '{:.4f}'
}))

## 5. Backtest James Check Signals

In [ ]:
# Define signals to test
signals_to_test = [
    # James Check signals
    Signal(metric='mvrv_sth', direction='below', threshold=1.0, regime='bull', name='JC_STH_MVRV<1_bull'),
    Signal(metric='sopr_sth', direction='below', threshold=1.0, regime='bull', name='JC_STH_SOPR<1_bull'),
    Signal(metric='mvrv_sth', direction='below', threshold=1.0, regime=None, name='JC_STH_MVRV<1_all'),
    Signal(metric='sopr_sth', direction='below', threshold=1.0, regime=None, name='JC_STH_SOPR<1_all'),
    
    # Existing approved signals for comparison
    Signal(metric='mvrv', direction='below', threshold=1.224, regime='bull', name='MVRV<20th_bull'),
    Signal(metric='mvrv_lth', direction='below', threshold=1.334, regime='bull', name='MVRV_LTH<20th_bull'),
]

print(f"Testing {len(signals_to_test)} signals...")

In [ ]:
# Run backtests with different exit modes
results = []

for exit_mode in [ExitMode.COMBINED, ExitMode.SIGNAL_EXIT]:
    config = BacktestConfig(
        exit_mode=exit_mode,
        hold_days=30,
        stop_loss=0.15,  # 15% stop loss
        max_hold_days=90  # Safety max hold
    )
    bt = Backtester(config)
    
    for signal in signals_to_test:
        try:
            result = bt.run(df_signals, signal, price_col='price')
            result_dict = result.to_dict()
            result_dict['exit_mode'] = exit_mode.value
            results.append(result_dict)
        except Exception as e:
            print(f"Error with {signal.name}: {e}")

results_df = pd.DataFrame(results)
print(f"\nCompleted {len(results_df)} backtest runs")

In [ ]:
# Display results
print("\n" + "=" * 90)
print("BACKTEST RESULTS COMPARISON")
print("=" * 90)

display_cols = ['signal_name', 'exit_mode', 'n_trades', 'total_return', 'sharpe_ratio', 
                'max_drawdown', 'win_rate', 'avg_trade_return', 'avg_hold_days']

results_display = results_df[display_cols].copy()
results_display = results_display.sort_values(['sharpe_ratio'], ascending=False)

display(results_display.style.format({
    'total_return': '{:.1%}',
    'sharpe_ratio': '{:.2f}',
    'max_drawdown': '{:.1%}',
    'win_rate': '{:.1%}',
    'avg_trade_return': '{:.1%}',
    'avg_hold_days': '{:.0f}'
}).background_gradient(subset=['sharpe_ratio'], cmap='RdYlGn'))

## 6. Combined James Check Signal Backtest

Test the combined signal (STH-MVRV < 1 AND STH-SOPR < 1)

In [ ]:
# Create a custom combined metric for backtesting
# jc_score: 2 when both conditions met, 1 when one met, 0 when none
df_signals['jc_score'] = (
    (df_signals['mvrv_sth'] < 1.0).astype(int) + 
    (df_signals['sopr_sth'] < 1.0).astype(int)
)

# Test the combined signal (jc_score == 2)
jc_combined_signal = Signal(
    metric='jc_score',
    direction='above',
    threshold=1.5,  # Must be > 1.5, so effectively == 2
    regime='bull',
    name='JC_Combined_bull'
)

# Also test without regime filter
jc_combined_all = Signal(
    metric='jc_score',
    direction='above',
    threshold=1.5,
    regime=None,
    name='JC_Combined_all'
)

# Run backtest
config = BacktestConfig(
    exit_mode=ExitMode.COMBINED,
    hold_days=30,
    stop_loss=0.15,
    max_hold_days=90
)
bt = Backtester(config)

result_combined_bull = bt.run(df_signals, jc_combined_signal, price_col='price')
result_combined_all = bt.run(df_signals, jc_combined_all, price_col='price')

print(result_combined_bull.summary())
print("\n" + "="*60)
print(result_combined_all.summary())

In [ ]:
# Visualize equity curves
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Get best results for comparison
config = BacktestConfig(exit_mode=ExitMode.COMBINED, stop_loss=0.15, max_hold_days=90)
bt = Backtester(config)

signals_to_plot = [
    Signal(metric='jc_score', direction='above', threshold=1.5, regime='bull', name='JC_Combined_bull'),
    Signal(metric='mvrv_sth', direction='below', threshold=1.0, regime='bull', name='JC_STH_MVRV<1_bull'),
    Signal(metric='mvrv', direction='below', threshold=1.224, regime='bull', name='MVRV<20th_bull'),
]

# Plot equity curves
ax1 = axes[0]
colors = ['green', 'cyan', 'orange']
for signal, color in zip(signals_to_plot, colors):
    result = bt.run(df_signals, signal, price_col='price')
    ax1.plot(result.equity_curve.index, result.equity_curve.values, 
             label=f"{signal.name} (Sharpe: {result.sharpe_ratio:.2f})", color=color, linewidth=1.5)

ax1.set_ylabel('Equity (Starting = 1.0)')
ax1.set_title('Equity Curves Comparison')
ax1.legend(loc='upper left')
ax1.grid(True, alpha=0.3)
ax1.set_yscale('log')

# Plot BTC price with signal markers
ax2 = axes[1]
ax2.plot(df_signals.index, df_signals['price'], color='white', alpha=0.7, linewidth=1)

# Mark JC Combined signals
jc_active = df_signals[df_signals['jc_core'] & (df_signals['price'] > df_signals.get('price_200d_sma', 0))]
ax2.scatter(jc_active.index, jc_active['price'], color='green', s=10, alpha=0.5, label='JC Buy Signal')

ax2.set_ylabel('BTC Price (USD)')
ax2.set_xlabel('Date')
ax2.set_title('BTC Price with James Check Buy Signals')
ax2.legend()
ax2.grid(True, alpha=0.3)
ax2.set_yscale('log')

plt.tight_layout()
plt.savefig('../data/results/jc_equity_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Trade Analysis

In [ ]:
# Detailed trade analysis for best signal
trades_df = result_combined_bull.get_trades_df()

print("\n" + "=" * 70)
print(f"TRADE ANALYSIS: {result_combined_bull.signal.name}")
print("=" * 70)

print(f"\nTotal Trades: {len(trades_df)}")
print(f"Winners: {(trades_df['return_pct'] > 0).sum()}")
print(f"Losers: {(trades_df['return_pct'] <= 0).sum()}")

# Show recent trades
print("\n--- Recent Trades ---")
display(trades_df.tail(10))

In [ ]:
# Trade return distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Return distribution
ax1 = axes[0]
returns = trades_df['return_pct'].dropna() * 100
ax1.hist(returns, bins=30, color='cyan', edgecolor='white', alpha=0.7)
ax1.axvline(returns.mean(), color='yellow', linestyle='--', linewidth=2, label=f'Mean: {returns.mean():.1f}%')
ax1.axvline(0, color='red', linestyle='-', linewidth=2)
ax1.set_xlabel('Trade Return (%)')
ax1.set_ylabel('Frequency')
ax1.set_title('Trade Return Distribution')
ax1.legend()

# Duration distribution
ax2 = axes[1]
durations = trades_df['duration_days'].dropna()
ax2.hist(durations, bins=30, color='orange', edgecolor='white', alpha=0.7)
ax2.axvline(durations.mean(), color='yellow', linestyle='--', linewidth=2, label=f'Mean: {durations.mean():.0f} days')
ax2.set_xlabel('Trade Duration (days)')
ax2.set_ylabel('Frequency')
ax2.set_title('Trade Duration Distribution')
ax2.legend()

plt.tight_layout()
plt.show()

## 8. Summary & Conclusions

In [ ]:
# Final summary comparison
print("\n" + "=" * 80)
print("FINAL COMPARISON: JAMES CHECK vs EXISTING SIGNALS")
print("=" * 80)

# Get best result for each signal type
combined_results = results_df[results_df['exit_mode'] == 'combined'].copy()

# Add JC Combined results
jc_combined_dict = result_combined_bull.to_dict()
jc_combined_dict['exit_mode'] = 'combined'
combined_results = pd.concat([combined_results, pd.DataFrame([jc_combined_dict])], ignore_index=True)

# Sort by Sharpe ratio
combined_results = combined_results.sort_values('sharpe_ratio', ascending=False)

print("\nRanked by Sharpe Ratio:\n")
print(f"{'Rank':<5} {'Signal':<30} {'Sharpe':>8} {'Return':>10} {'Win Rate':>10} {'Trades':>8}")
print("-" * 80)

for i, (_, row) in enumerate(combined_results.iterrows(), 1):
    print(f"{i:<5} {row['signal_name']:<30} {row['sharpe_ratio']:>8.2f} "
          f"{row['total_return']*100:>9.1f}% {row['win_rate']*100:>9.1f}% {row['n_trades']:>8}")

print("\n" + "=" * 80)
print("KEY FINDINGS")
print("=" * 80)

best = combined_results.iloc[0]
print(f"""
1. Best performing signal: {best['signal_name']}
   - Sharpe Ratio: {best['sharpe_ratio']:.2f}
   - Total Return: {best['total_return']*100:.1f}%
   - Win Rate: {best['win_rate']*100:.1f}%

2. James Check signals {'outperform' if 'JC' in best['signal_name'] else 'underperform'} existing signals

3. The combined signal (STH-MVRV < 1 AND STH-SOPR < 1) provides better
   signal quality through confluence filtering.

4. Bull market regime filter improves signal quality by avoiding
   bear market false positives.
""")

In [ ]:
# Save results
output_path = Path('../data/results/james_check_analysis.json')
import json

analysis_results = {
    'timestamp': datetime.now().isoformat(),
    'signals_tested': len(combined_results),
    'best_signal': best['signal_name'],
    'best_sharpe': float(best['sharpe_ratio']),
    'results': combined_results.to_dict(orient='records')
}

with open(output_path, 'w') as f:
    json.dump(analysis_results, f, indent=2, default=str)

print(f"\n✓ Results saved to {output_path}")

## 9. Current Market Status

In [ ]:
# Current market conditions
latest = df_signals.iloc[-1]

print("\n" + "=" * 60)
print(f"CURRENT MARKET STATUS ({latest.name.date()})")
print("=" * 60)

print(f"""
BTC Price:          ${latest['price']:,.0f}
STH Realized Price: ${latest['realized_price_sth']:,.0f}

JAMES CHECK METRICS:
  STH-MVRV:         {latest['mvrv_sth']:.3f} {'< 1.0 ✓ BUY' if latest['mvrv_sth'] < 1 else '>= 1.0'}
  STH-SOPR:         {latest['sopr_sth']:.3f} {'< 1.0 ✓ BUY' if latest['sopr_sth'] < 1 else '>= 1.0'}

SIGNAL STATUS:
  JC Core:          {'🟢 ACTIVE' if latest['jc_core'] else '⚫ INACTIVE'}
  JC Enhanced:      {'🟢 ACTIVE' if latest['jc_enhanced'] else '⚫ INACTIVE'}
  Bull Filter:      {'✓ Above 200 SMA' if latest['bull_filter'] else '✗ Below 200 SMA'}

COMPARISON METRICS:
  MVRV:             {latest['mvrv']:.3f}
  NUPL:             {latest['nupl']:.3f}
""")

if latest['jc_enhanced']:
    print("⚠️  JAMES CHECK BUY-THE-DIP SIGNAL IS ACTIVE!")
else:
    print("No active buy signal currently.")